# L5 综合实验｜可测试的量化任务执行器

本实验整合第 51–58 课：计时与重试 → 受控并发 → 数值热点 JIT → 不变量 → pytest → 版本与容器配置清单。

- 环境：`pkm-quant`
- 数据：离线模拟任务
- 输出：`outputs/L5_engineering_audit.json`

## 验收

1. 暂时失败可重试且次数可见；
2. 异步并发有 Semaphore 上限；
3. Numba 与 Python 参考结果一致；
4. 无未来信息和成本单调性由断言保护；
5. 产物记录依赖版本、数据哈希和部署配置状态。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# 导入工程实验所需的标准库和 Numba。
import asyncio
import hashlib
import importlib.metadata as metadata
import time
from numba import njit

## 1. 受控并发任务保留逐项状态

In [3]:
# 并发执行模拟数据任务，最多 3 个同时在途。
semaphore = asyncio.Semaphore(3)
state = {"in_flight": 0, "max_in_flight": 0}

async def task(symbol):
    async with semaphore:
        state["in_flight"] += 1
        state["max_in_flight"] = max(state["max_in_flight"], state["in_flight"])
        start = time.perf_counter()
        try:
            await asyncio.sleep(0.01)
            if symbol == "FAIL":
                raise TimeoutError("simulated")
            return {"symbol": symbol, "status": "ok", "elapsed_ms": (time.perf_counter() - start) * 1000}
        except TimeoutError as exc:
            return {"symbol": symbol, "status": "failed", "error": str(exc)}
        finally:
            state["in_flight"] -= 1

symbols = ["A", "B", "C", "D", "FAIL", "E"]
statuses = pd.DataFrame(await asyncio.gather(*(task(symbol) for symbol in symbols)))
display(statuses)
assert state["max_in_flight"] == 3
assert statuses["status"].value_counts().to_dict() == {"ok": 5, "failed": 1}

,symbol,status,elapsed_ms,error
0,A,ok,12.070417,NaN
1,B,ok,12.069083,NaN
2,C,ok,12.068833,NaN
3,D,ok,10.707125,NaN
4,FAIL,failed,NaN,simulated
5,E,ok,10.770375,NaN


## 2. 数值热点保持参考实现与 JIT 实现

In [4]:
# 编译组合收益累积循环，并与 NumPy 参考结果逐项比较。
def nav_python(returns):
    output = np.empty(len(returns))
    value = 1.0
    for i, item in enumerate(returns):
        value *= 1.0 + item
        output[i] = value
    return output

@njit
def nav_numba(returns):
    output = np.empty(len(returns))
    value = 1.0
    for i in range(len(returns)):
        value *= 1.0 + returns[i]
        output[i] = value
    return output

rng = np.random.default_rng(365)
returns = rng.normal(0.0002, 0.01, 100_000)
reference = nav_python(returns)
jitted = nav_numba(returns)
np.testing.assert_allclose(jitted, reference, rtol=1e-12, atol=1e-12)
print("最终净值:", jitted[-1], "编译签名:", nav_numba.signatures)

最终净值: 49546453.15741527 编译签名: [(Array(float64, 1, 'C', False, aligned=True),)]


## 3. 量化不变量与可追溯清单

In [5]:
# 验证成本单调性，并保存工程质量审计。
signal = (returns > 0).astype(float)
position = np.r_[0.0, signal[:-1]]
gross = position * returns
turnover = np.abs(np.diff(np.r_[0.0, position]))
net = gross - 0.001 * turnover
assert np.prod(1 + net) <= np.prod(1 + gross) + 1e-12

data_path = DATA_DIR / "daily_prices.csv"
audit = {
    "lab": "L5",
    "async_max_in_flight": state["max_in_flight"],
    "task_status": statuses.fillna("").to_dict(orient="records"),
    "numba_matches_reference": bool(np.allclose(jitted, reference)),
    "cost_monotonicity": True,
    "data_sha256": hashlib.sha256(data_path.read_bytes()).hexdigest(),
    "versions": {
        package: metadata.version(package)
        for package in ["numpy", "numba", "pytest"]
    },
    "git_manifest": "57_experiment_manifest.json",
    "dockerfile": "58_Dockerfile.example",
}
audit_path = OUTPUT_DIR / "L5_engineering_audit.json"
audit_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(pd.Series({k: v for k, v in audit.items() if k != "task_status"}, name="value").to_frame())
assert audit_path.exists()

,value
lab,L5
async_max_in_flight,3
numba_matches_reference,True
cost_monotonicity,True
data_sha256,e57dd4377afd30cec702ad33494e0a98d2a5b83c8aeccc...
versions,"{'numpy': '2.4.6', 'numba': '0.66.0', 'pytest'..."
git_manifest,57_experiment_manifest.json
dockerfile,58_Dockerfile.example


## 方法边界与复用结论


- L5 验证的是可测试工程骨架，不是生产调度系统。
- 并发、JIT 和容器只有在正确性测试通过后才应引入。
- 外部接口、部署、密钥和自动发布仍需独立权限与监控设计。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。